## Integer Programming: Assign Firetrucks To Districts

A city has $m$ firetrucks and wants to assign each one to one of $m$ of the districts. For each district, the population-weighted firefighting time is defined as the product of the district population times the amount of time it takes for the closest firetruck to travel to it. We want to assign the $m$ firetrucks to minimize the maximum population-weighted firefighting time among all districts.

This can be formulated as:

$$
\begin{array}{rlll}
    \min & w & \\
    \text{s.t.}
        & \displaystyle \sum_{f \in D} e_{ft} = 1 & \forall t \in D & \text{exactly 1 incoming edge per district} \\[15pt]
        & \displaystyle \sum_{i \in D} x_{i} = m & & \text{exactly m assigned firetrucks} \\[15pt]
        & e_{ft} \le x_f & \forall f,t \in D & \text{edge requires firetruck at origin} \\[5pt]
        & w \geq e_{ft} d_{ft} p_t & \forall f,t \in D & \text{max firefighting time >= each firefighting time} \\[5pt]
        & x_{i} \in \{0, 1\} &\forall i \in D & \text{assign 0 or 1 firetrucks per location} \\[5pt]
        & e_{ft} \in \{0, 1\} &\forall f,t \in D & \text{each edge is either used (1) or not used (0)}
\end{array}
$$

with variables defined as follows:
- $w$: the maximum population-weighted firefighting time
- $x_i$: whether district $i$ has a firetruck
- $d_{ft}$: the travel time from district $f$ to district $t$
- $e_{ft}$: whether a firetruck travels from district $f$ to district $t$
- $p_{i}$: the population of district $i$
- $D$: the set of districts
- $m$: the number of firetrucks to be assigned

Problem 2 of final (week 6) quiz for [Operations Research (2): Optimization Algorithms](https://www.coursera.org/learn/operations-research-algorithms/home/welcome).


In [1]:
# Imports and input data
import numpy as np
import pandas as pd
from scipy.optimize import LinearConstraint, milp

district_travel_times = pd.DataFrame([
    (0, 3, 4, 6, 8, 9, 8, 10),
    (3, 0, 5, 4, 8, 6, 12, 9),
    (4, 5, 0, 2, 2, 3, 5, 7),
    (6, 4, 2, 0, 3, 2, 5, 4),
    (8, 8, 2, 3, 0, 2, 2, 4),
    (9, 6, 3, 2, 2, 0, 3, 2),
    (8, 12, 5, 5, 2, 3, 0, 2),
    (10, 9, 7, 4, 4, 2, 2, 0),
])

district_populations = [
    40,
    30,
    35,
    20,
    15,
    50,
    45,
    60,
]

firetrucks = 2

print("Travel times between districts i and j:")
display(district_travel_times)

print("District populations:", ", ".join(str(x) for x in district_populations))

print("Number of firetrucks to assign:", firetrucks)

Travel times between districts i and j:


,0,1,2,3,4,5,6,7
0,0,3,4,6,8,9,8,10
1,3,0,5,4,8,6,12,9
2,4,5,0,2,2,3,5,7
3,6,4,2,0,3,2,5,4
4,8,8,2,3,0,2,2,4
5,9,6,3,2,2,0,3,2
6,8,12,5,5,2,3,0,2
7,10,9,7,4,4,2,2,0


District populations: 40, 30, 35, 20, 15, 50, 45, 60
Number of firetrucks to assign: 2


In [2]:
# Set objective and constraints

districts = len(district_populations)

# Variables:
# [
#    district 0 firetruck, district 1 firetruck, ..., district 7 firetruck,
#    whether we travel from district 0 to 0, from 0 to 1, ..., from 1 to 0, ..., from 7 to 7,
#    max population-weighted firefighting time
# ]

# Only minimise max firefighting time
coef = [0] * districts + [0] * districts * districts + [1]

# Convenience function to get index of variable for travelling from one district to another
def travel(from_district, to_district):
    return districts + districts * from_district + to_district

def get_constraints():
    var_count = len(coef)

    constraints = []

    # Each district must have exactly one incoming edge (sum of edges = 1)
    for to_district in range(districts):
        A_curr = np.zeros(var_count)
        for from_district in range(districts):
            A_curr[travel(from_district, to_district)] = 1
        constraints.append(LinearConstraint(A_curr, 1, 1))

    # An edge may only be set if its from district has a firetruck (firetruck >= edge)
    # This can be represented with a batch constraint as:
    # (M+1) * firetruck >= sum of M edges, or (M+1) * firetruck - sum of M edges >= 0
    for from_district in range(districts):
        A_curr = np.zeros(var_count)
        A_curr[from_district] = districts + 1
        for to_district in range(districts):
            A_curr[travel(from_district, to_district)] = -1
        constraints.append(LinearConstraint(A_curr, lb=0))

    # Set max firefighting time
    # max firefighting time >= each travel time, i.e. max firefighting time - each travel time >= 0
    # Since there's only 1 travel time per to-district, sum(travel time) = max(travel time) for some to-district,
    # so we can use the sum to only have a constraint for each to-district, rather than having one for each from-to pair
    for to_district in range(districts):
        A_curr = np.zeros(var_count)
        A_curr[-1] = 1
        for from_district in range(districts):
            A_curr[travel(from_district, to_district)] = -district_travel_times[from_district][to_district] * district_populations[to_district]
        constraints.append(LinearConstraint(A_curr, lb=0))

    # Sum of assigned firetrucks shouldn't exceed number of firetrucks
    A_curr = np.zeros(var_count)
    A_curr[:districts] = 1
    constraints.append(LinearConstraint(A_curr, firetrucks, firetrucks))

    # No need to ensure edge values are <= 1, since a value of 1 always has a smaller objective value than a value of more than 1

    # Make sure we only assign 0 or 1 firetrucks per location
    for district in range(districts):
        A_curr = np.zeros(var_count)
        A_curr[district] = 1
        constraints.append(LinearConstraint(A_curr, 0, 1))
    
    return constraints

In [3]:
# Run solver

# Set all variables to be integers
integrality = np.ones_like(coef)

res = milp(c=coef, integrality=integrality, constraints=get_constraints())
res

        message: Optimization terminated successfully. (HiGHS Status 7: Optimal)
        success: True
         status: 0
            fun: 135.0
              x: [ 0.000e+00  1.000e+00 ...  0.000e+00  1.350e+02]
 mip_node_count: 1
 mip_dual_bound: 135.0
        mip_gap: 0.0

In [4]:
# Print solution details
print("Firetrucks at districts:", ", ".join([str(i) for i in range(districts) if res.x[i]]))
print("Maximum population-weighted travel cost:", res.x[-1].round())
print()

pd.DataFrame({
        "from district": from_district,
        "to district": to_district,
        "travel time": district_travel_times[from_district][to_district],
        "population": district_populations[to_district],
        "population-weighted travel cost": district_travel_times[from_district][to_district] * district_populations[to_district],
    }
    for to_district in range(districts)
    for from_district in range(districts)
    if res.x[travel(from_district, to_district)]
).style.hide()  # hide index

Firetrucks at districts: 1, 5
Maximum population-weighted travel cost: 135.0



from district,to district,travel time,population,population-weighted travel cost
1,0,3,40,120
1,1,0,30,0
5,2,3,35,105
5,3,2,20,40
1,4,8,15,120
5,5,0,50,0
5,6,3,45,135
5,7,2,60,120


## Greedy heuristic

Assign a firetruck to a district that doesn't currently have a firetruck, and which would minimize the maximum population-weighted firefighting times among all districts. If there are multiple districts satisfying this conditions, pick the one with the smallest district ID. Repeat until all firetrucks have been assigned.

Use 3 firetrucks instead of 2.

Problem 3 of final (week 6) quiz for [Operations Research (2): Optimization Algorithms](https://www.coursera.org/learn/operations-research-algorithms/home/welcome).

In [5]:
# Update number of firetrucks
firetrucks = 3

In [7]:
# Run solver

# Set all variables to be integers
integrality = np.ones_like(coef)

res = milp(c=coef, integrality=integrality, constraints=get_constraints())
res

        message: Optimization terminated successfully. (HiGHS Status 7: Optimal)
        success: True
         status: 0
            fun: 100.0
              x: [ 1.000e+00  0.000e+00 ...  1.000e+00  1.000e+02]
 mip_node_count: 1
 mip_dual_bound: 100.0
        mip_gap: 0.0

In [ ]:
# Run heuristic

assigned_districts = np.zeros(districts)

while sum(assigned_districts) < firetrucks:
    best_district = (None, np.inf)
    # Look for the best new district to assign a firetruck to
    for new_district in range(districts):
        if assigned_districts[new_district] == 1:
            continue

        assigned_districts[new_district] = 1

        # For each district, find the closest district with a firetruck
        # Then use that to find the district with the maximum cost
        max_travel_cost = 0
        for to_district in range(districts):
            travel_costs = assigned_districts * district_travel_times[to_district] * district_populations[to_district]
            max_travel_cost = max(max_travel_cost, min(travel_costs[assigned_districts != 0]))
        if max_travel_cost < best_district[1]:
            best_district = (new_district, max_travel_cost)

        assigned_districts[new_district] = 0

    print("Assigning to district", best_district[0], "with max travel cost", best_district[1])
    assigned_districts[best_district[0]] = 1

print()
print("Assigned districts:", ", ".join(str(i) for i in range(districts) if assigned_districts[i]))
print("Maximum travel cost:", best_district[1])
print("Optimal travel cost:", res.fun)
print(f"Optimality gap: {(res.fun - best_district[1]) / res.fun:.2%}")

pd.DataFrame({
        "from district": from_district,
        "to district": to_district,
        "travel time": district_travel_times[from_district][to_district],
        "population": district_populations[to_district],
        "population-weighted travel cost": district_travel_times[from_district][to_district] * district_populations[to_district],
    }
    for to_district in range(districts)
    for from_district in range(districts)
    if from_district == district_travel_times[to_district][assigned_districts != 0].idxmin()
).style.hide()  # hide index

Assigning to district 3 with max travel cost 240.0
Assigning to district 0 with max travel cost 240.0
Assigning to district 7 with max travel cost 100.0

Assigned districts: 0, 3, 7
Maximum travel cost: 100.0
Optimal travel cost: 100.0
Optimality gap: 0.00%


from district,to district,travel time,population,population-weighted travel cost
0,0,0,40,0
0,1,3,30,90
3,2,2,35,70
3,3,0,20,0
3,4,3,15,45
3,5,2,50,100
7,6,2,45,90
7,7,0,60,0
